1. StrOutputParser
2. CommaSeparatedListOutputParser
3. PydanticOutputParser
4. JsonOutputParser
5. with_structured_output

In [ ]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm_groq = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
messages = [
    ("system","You are an assistant and your anme is {name}"),
    ("human","Hi, how are you doing?"),
    ("ai","Doing good, how can I help you?"),
    ("human","what is attention mechanism?")
    # ("palceholder","{history}")
]

chat_template = ChatPromptTemplate.from_messages(messages = messages)
# chat_ans = chat_template.invoke({"name":"ai assistant"})

#llm 



# stroutput parser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = chat_template | llm_groq | StrOutputParser()
answer = chain.invoke({"name":"ai assistant"})
print(answer)

# comma seperated parser

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

messages = [
    ("system","You are designed for giving cricketers list.\n{format_instructions}"),
    ("human","Give me the list of 10 {country} cricketers.")
]

parser = CommaSeparatedListOutputParser()
chat_template = ChatPromptTemplate.from_messages(messages = messages).partial(format_instructions = parser.get_format_instructions() )

chain = chat_template | llm_groq
answer = chain.invoke({"country":"Australia"})
print(answer.content)

PydanticOutputParser

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class Cricketer(BaseModel):
    name : str = Field(description="Name of cricketer")
    age : int = Field(description="age of cricketer")
    country : str = Field(description="Name of country for which the cricketer is playing")
    ipl_team : str = Field(description="Name of the ipl team for which cricketer is playing for")

parser = PydanticOutputParser(pydantic_object=Cricketer)

chat_template = ChatPromptTemplate.from_messages([
    ("system","Give the cricketer information in JSON format ONLY, no MARK UP or any extra information.\n{format_instructions}"),
    ("human","Please give the details of {cricketername} in JSON format ONLY.")
]).partial(format_instructions= parser.get_format_instructions)

chain = chat_template | llm_groq 
answer = chain.invoke({"cricketername":"Virat Kohli"})
print(answer.content)

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
class CricketersList(BaseModel):
    playersList : list[Cricketer]

parser = JsonOutputParser(pydantic_object=CricketersList)
chat_template = ChatPromptTemplate.from_messages([
    ("system","Give the cricketer information in JSON format ONLY, no MARK UP or any extra information.\n{format_instructions}"),
    ("human","Please give the details of  10 {country} cricketers in JSON format ONLY.")
]).partial(format_instructions= parser.get_format_instructions)

chain = chat_template | llm_groq | parser

for chunks in chain.stream({"country":"Indian"}):
    print(chunks)




with_structured_output

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

class Cricketer(BaseModel):
    name : str = Field(description="Name of cricketer")
    age : int = Field(description="age of cricketer")
    country : str = Field(description="Name of country for which the cricketer is playing")
    ipl_team : str = Field(description="Name of the ipl team for which cricketer is playing for")

chat_template = ChatPromptTemplate.from_messages([
    ("system","Give the cricketer information in JSON format ONLY, no MARK UP or any extra information.\n"),
    ("human","Please give the details of {cricketername} in JSON format ONLY.")
])

llm_structured_output = llm_groq.with_structured_output(schema=Cricketer)
chain = chat_template | llm_structured_output
answer = chain.invoke({"cricketername":"Virat Kohli"})
print(answer)
